# WattMosaic

### Campus energy forecasting with incomplete sensors

This public walkthrough accompanies the project reported for NTU Datathon / Deep Learning Week 2026, Track 1. It demonstrates four-sensor missingness masks and simplified core/specialist routing.

**Project-reported public leaderboard RMSE:** `3.17093`. The event score and development metrics below are archived values, not results recomputed by this notebook. No independent leaderboard record or trained artifacts are included.

Competition datasets, trained weights and the full training/export pipeline are not redistributed. The data-free check below verifies public routing behavior; it does not evaluate forecast accuracy.

## 1. Problem framing

The target is continuous campus-facility energy consumption. Useful predictors include building identity/type, hour, weekday, month, temperature, humidity, occupancy, and previous usage.

A major practical challenge was **distribution shift in sensor missingness**: the reference/test features contained different missing-sensor combinations from the labelled training data. That motivated a mask-aware model rather than a single generic imputation path.


In [1]:
SENSORS = ['temperature', 'humidity', 'occupancy', 'previous_usage']

def missingness_mask(frame):
    """Encode the four sensor availability states as an integer 0..15."""
    import numpy as np
    return frame[SENSORS].isna().to_numpy(dtype=np.int64) @ (2 ** np.arange(len(SENSORS)))

# Four binary sensor states => 16 routing patterns.
assert 2 ** len(SENSORS) == 16


## 2. Core and specialist routing

The project notes describe a shared energy core and 15 specialists for non-empty missing-sensor masks. Complete rows use the core directly; incomplete rows combine the core with the matching specialist.

The function below illustrates that routing interface. Its callbacks stand in for models; this public notebook does not train a domain-aware specialist or contain fitted tree inference. Claims about transfer to the event leaderboard cannot be reproduced from these callbacks alone.

In [2]:
def route_prediction(core_predict, specialist_predict, frame):
    """Simplified version of WattMosaic's mask-aware inference routing."""
    import numpy as np
    masks = missingness_mask(frame)
    out = np.empty(len(frame), dtype=float)

    for mask in np.unique(masks):
        idx = np.flatnonzero(masks == mask)
        subset = frame.iloc[idx]
        base = core_predict(subset, int(mask))
        if mask == 0:
            out[idx] = base
        else:
            out[idx] = base + specialist_predict(subset, int(mask))

    return np.maximum(out, 0.0)


### Data-free routing check

Construct each of the 16 missingness patterns with synthetic values. Fixed callbacks verify mask selection and output order. These values are illustrative test inputs, not campus measurements or trained forecasts.

In [3]:
import numpy as np
import pandas as pd

synthetic_masks = np.arange(16)
synthetic = pd.DataFrame({
    sensor: [np.nan if mask & (1 << bit) else float(bit + 1) for mask in synthetic_masks]
    for bit, sensor in enumerate(SENSORS)
})
np.testing.assert_array_equal(missingness_mask(synthetic), synthetic_masks)
synthetic_predictions = route_prediction(
    lambda subset, mask: np.full(len(subset), 10.0),
    lambda subset, mask: np.full(len(subset), float(mask)),
    synthetic,
)
np.testing.assert_array_equal(synthetic_predictions, 10.0 + synthetic_masks)
print("All 16 synthetic routing patterns passed; trained forecast accuracy is not evaluated.")

All 16 synthetic routing patterns passed; trained forecast accuracy is not evaluated.


## 3. Reported evaluation context

Project notes report development checks under ordinary and test-like joint missingness, separately from public leaderboard feedback. The following table preserves those reported values. Running this cell constructs a table; it does not load observations, compute metrics or validate a leaderboard submission.

In [4]:
import pandas as pd

results = pd.DataFrame([
    {'evaluation': 'Development audit — natural missingness', 'RMSE': 3.022861, 'MAE': 2.381042, 'R2': 0.973150},
    {'evaluation': 'Development audit — test-like joint missingness', 'RMSE': 3.214310, 'MAE': 2.507000, 'R2': 0.969600},
    {'evaluation': 'Hackathon public leaderboard', 'RMSE': 3.170930, 'MAE': None, 'R2': None},
])
results


,evaluation,RMSE,MAE,R2
0,Development audit — natural missingness,3.022861,2.381042,0.97315
1,Development audit — test-like joint missingness,3.214310,2.507000,0.96960
2,Hackathon public leaderboard,3.170930,NaN,NaN


## 4. Archived event runtime

Project notes describe portable tree export and output-validation work for a fixed Python 3.12 competition environment. The export implementation, fitted trees and submission checks are not included here, so numerical equivalence and trained-model behavior cannot be verified from this release.

`EXPECTED_RUNTIME` records the event environment. `requirements.txt` preserves those archival pins; `requirements-smoke.txt` supplies a separate current environment for the public walkthrough checks.

In [5]:
EXPECTED_RUNTIME = {
    'python': '3.12',
    'joblib': '1.4.2',
    'lightgbm': '4.5.0',
    'matplotlib': '3.9.2',
    'numpy': '2.1.3',
    'pandas': '2.2.3',
    'scikit-learn': '1.5.2',
    'scipy': '1.14.1',
    'seaborn': '0.13.2',
    'xgboost': '2.1.3',
}
EXPECTED_RUNTIME


{'python': '3.12',
 'joblib': '1.4.2',
 'lightgbm': '4.5.0',
 'matplotlib': '3.9.2',
 'numpy': '2.1.3',
 'pandas': '2.2.3',
 'scikit-learn': '1.5.2',
 'scipy': '1.14.1',
 'seaborn': '0.13.2',
 'xgboost': '2.1.3'}

## 5. Public-release boundaries

This notebook is intentionally a **portfolio-oriented technical walkthrough**, not the raw competition workspace.

Excluded from the public repository:

- organizer-provided training/test datasets and submission template;
- serialized final model weights;
- generated leaderboard prediction CSVs;
- intermediate Round 1–7 experiments and probing artifacts;
- internal validation dumps and submission checklists.

The original materials did not explicitly grant public redistribution rights for the supplied datasets, so those files are not republished here.


## 6. Takeaways

1. **Missingness is part of the prediction problem.** Different sensor failures can warrant different experts.
2. **Better imputation is not automatically better downstream prediction.** Optimize the task metric.
3. **Validation should resemble deployment.** Test-like missingness checks were more informative than generic CV alone.
4. **Complexity must earn its place.** Several larger boosting and ensemble variants were rejected when they failed to transfer.
5. **Reproducibility is a modelling feature.** Runtime pinning, portable inference, and defensive output checks mattered under competition constraints.
